In [0]:
from pyspark.sql.types import StructType, StringType, StructField, DecimalType, IntegerType
from pyspark.sql import functions as F

In [0]:
catalogue_name = 'ecommerce'

###Brands

In [0]:
brand_schema = StructType([
    # StructField(name, dataType, nullable=True, metadata=None)
    StructField("brand_code",StringType(),False),
    StructField("brand_name",StringType(),True),
    StructField("category_code",StringType(),True),
])

raw_data = "/Volumes/ecommerce/source_data/raw/brands/*.csv" 

df = spark.read.option("delimeter", ",").csv(raw_data , schema=brand_schema, header=True)
# _metadata.filepath: gives the location of file
df = df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())
display(df.limit(5))

# Write the data to table called delata table 
# mergeSchema=true means it allows you to update the schema of table later in the developement

df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brz_brands")

###Category

In [0]:
# StructField(name, dataType, nullable=True, metadata=None)

category_schema = StructType(
    [ StructField("category_code",StringType(),False),
     StructField("category_name",StringType(),False)]
)

raw_data = "/Volumes/ecommerce/source_data/raw/category/*.csv"

df = spark.read.option("delimeter",",").option("header","true").schema(category_schema).csv(raw_data)
df = df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())
display(df.limit(5))

# Save to Bronze

df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brnz_category")
    

category_code,category_name,_source_file,_ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-07-26T12:23:30.816Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-07-26T12:23:30.816Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-07-26T12:23:30.816Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-07-26T12:23:30.816Z
bks,Books,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv,2026-07-26T12:23:30.816Z


###Customer

In [0]:
customer_schema = StructType([
    StructField("id",StringType(),False),
    StructField("phone",StringType(),True),
    StructField("country_code",StringType(),True),
    StructField("country",StringType(),True),
    StructField("state",StringType(),True)
])

raw_data ="/Volumes/ecommerce/source_data/raw/customers/*.csv"

df = spark.read.option("delimeter",",").option("header","true").schema(customer_schema).csv(raw_data)
df = df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())
display(df.limit(5))

# Save to bronze

df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brnz_customer")  


id,phone,country_code,country,state,_source_file,_ingested_at
CUST000000000001,917280033536.0,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:13.667Z
CUST000000000002,619489725433.0,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:13.667Z
CUST000000000003,919390066524.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:13.667Z
CUST000000000004,917073741793.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:13.667Z
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/source_data/raw/customers/customers.csv,2026-08-05T07:50:13.667Z


###Date

In [0]:
date_schema = StructType([
    StructField("date",StringType()),
    StructField("year",IntegerType()),
    StructField("day_name",StringType()),
    StructField("quarter",IntegerType()),
    StructField("week_of_year",IntegerType())

])

raw_data = "/Volumes/ecommerce/source_data/raw/date/*.csv"

df = spark.read.option("delimiter",",").option("header","true").schema(date_schema).csv(raw_data)
df = df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())
display(df.limit(5))

df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brnz_date")



date,year,day_name,quarter,week_of_year,_source_file,_ingested_at
01-08-2025,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:05:58.542Z
02-08-2025,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:05:58.542Z
03-08-2025,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:05:58.542Z
04-08-2025,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:05:58.542Z
05-08-2025,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date/date.csv,2026-07-26T13:05:58.542Z


In [0]:
product_schema = StructType([
    StructField("product_id",StringType(),False),
    StructField("ske",StringType(),True),
    StructField("category_code",StringType(),True),
    StructField("brand_code",StringType(),True),
    StructField("colour",StringType(),True),
    StructField("size",StringType(),True),
    StructField("material",StringType(),True),
    StructField("weight_grams",StringType(),True),
    StructField("length_cm",StringType(),True),
    StructField("width_cm",DecimalType(),True),
    StructField("height_cm",DecimalType(),True),
    StructField("rating_count",IntegerType(),True)

])

raw_data = "/Volumes/ecommerce/source_data/raw/products/*.csv"

df = spark.read.options(delimeter=",",header="true").schema(product_schema).csv(raw_data)
df = df.withColumn("_source_file",F.col("_metadata.file_path"))\
    .withColumn("_ingested_at",F.current_timestamp())
display(df.limit(5))

df.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalogue_name}.bronze.brnz_product")



product_id,ske,category_code,brand_code,colour,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,_source_file,_ingested_at
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17,6,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-07-26T13:15:27.457Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12,4,1,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-07-26T13:15:27.457Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",14,4,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-07-26T13:15:27.457Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",5,6,50,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-07-26T13:15:27.457Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",16,7,-4,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-07-26T13:15:27.457Z
